# Kapittel 7: Bygging av chatapplikasjoner
## Microsoft Foundry Models API Raskstart

Denne notatboken er tilpasset fra [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) som inkluderer notatbøker som får tilgang til [Azure OpenAI](notebook-azure-openai.ipynb) tjenester.

> **Merk:** GitHub Models fases ut ved slutten av juli 2026. Denne notatboken retter seg nå mot [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), som tilbyr samme gratis prøveperiode for modellkatalog og Azure AI Inference SDK-opplevelse.


# Oversikt  
"Store språkmodeller er funksjoner som kartlegger tekst til tekst. Gitt en inndatatekststreng, prøver en stor språkmodell å forutsi teksten som kommer neste"(1). Denne "hurtigstart"-notatboken vil introdusere brukere for høynivå LLM-konsepter, kjernepakke-krav for å komme i gang med AML, en myk introduksjon til promptdesign, og flere korte eksempler på ulike bruksområder. 


## Innholdsfortegnelse  

[Oversikt](#overview)  
[Hvordan bruke OpenAI-tjenesten](#how-to-use-openai-service)  
[1. Opprette din OpenAI-tjeneste](#1.-creating-your-openai-service)  
[2. Installasjon](#2.-installation)    
[3. Legitimationsopplysninger](#3.-credentials)  

[Brukstilfeller](#use-cases)    
[1. Oppsummere tekst](#1.-summarize-text)  
[2. Klassifisere tekst](#2.-classify-text)  
[3. Generere nye produktnavn](#3.-generate-new-product-names)  
[4. Fininnstille en klassifiserer](#4.fine-tune-a-classifier)  

[Referanser](#references)


### Bygg din første prompt  
Denne korte øvelsen vil gi en grunnleggende introduksjon til å sende inn prompts til en modell i Microsoft Foundry Models for en enkel oppgave "oppsummering".


**Steg**:  
1. Installer `azure-ai-inference`-biblioteket i ditt Python-miljø, hvis du ikke allerede har gjort det.  
2. Last inn standard hjelpebiblioteker og sett opp dine Microsoft Foundry Models-legitimasjoner.  
3. Velg en modell for oppgaven din  
4. Lag en enkel prompt for modellen  
5. Send forespørselen til modellens API!


### 1. Installer `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importer hjelpbiblioteker og oppretter legitimasjon


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Finne riktig modell  
Modeller som GPT-4o og GPT-4o mini kan forstå og generere naturlig språk, og er tilgjengelige i Microsoft Foundry Models-katalogen sammen med modeller fra Meta, Mistral, Cohere og Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Prompt Design  

"The magic of large language models is that by being trained to minimize this prediction error over vast quantities of text, the models end up learning concepts useful for these predictions. For example, they learn concepts like"(1):

* how to spell
* how grammar works
* how to paraphrase
* how to answer questions
* how to hold a conversation
* how to write in many languages
* how to code
* etc.

#### How to control a large language model  
"Of all the inputs to a large language model, by far the most influential is the text prompt(1).

Large language models can be prompted to produce output in a few ways:

Instruction: Tell the model what you want
Completion: Induce the model to complete the beginning of what you want
Demonstration: Show the model what you want, with either:
A few examples in the prompt
Many hundreds or thousands of examples in a fine-tuning training dataset"



#### There are three basic guidelines to creating prompts:

**Show and tell**. Make it clear what you want either through instructions, examples, or a combination of the two. If you want the model to rank a list of items in alphabetical order or to classify a paragraph by sentiment, show it that's what you want.

**Provide quality data**. If you're trying to build a classifier or get the model to follow a pattern, make sure that there are enough examples. Be sure to proofread your examples — the model is usually smart enough to see through basic spelling mistakes and give you a response, but it also might assume this is intentional and it can affect the response.

**Check your settings.** The temperature and top_p settings control how deterministic the model is in generating a response. If you're asking it for a response where there's only one right answer, then you'd want to set these lower. If you're looking for more diverse responses, then you might want to set them higher. The number one mistake people make with these settings is assuming that they're "cleverness" or "creativity" controls.


Source: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Send inn!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Gjenta det samme kallet, hvordan sammenlignes resultatene?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Oppsummer Tekst  
#### Utfordring  
Oppsummer tekst ved å legge til 'tl;dr:' på slutten av et tekstavsnitt. Legg merke til hvordan modellen forstår å utføre en rekke oppgaver uten ekstra instruksjoner. Du kan eksperimentere med mer beskrivende prompt enn tl;dr for å endre modellens oppførsel og tilpasse oppsummeringen du mottar(3).  

Nyere arbeid har vist betydelige forbedringer på mange NLP-oppgaver og benchmarks ved å forhåndstrene på et stort tekstkorpus etterfulgt av finjustering på en spesifikk oppgave. Selv om arkitekturen vanligvis er oppgave-agnostisk, krever denne metoden fortsatt oppgavespesifikke finjusteringsdatasett på tusenvis eller titusenvis av eksempler. Til sammenligning kan mennesker vanligvis utføre en ny språkoppgave med bare noen få eksempler eller fra enkle instruksjoner – noe som nåværende NLP-systemer fortsatt i stor grad sliter med. Her viser vi at skalering av språkmodeller i stor grad forbedrer oppgave-agnostisk, få-skudd ytelse, noen ganger til og med oppnår konkurransedyktighet med tidligere state-of-the-art finjusteringsmetoder. 



Tl;dr


# Øvelser for flere brukstilfeller  
1. Oppsummer tekst  
2. Klassifiser tekst  
3. Generer nye produktnavn


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klassifiser tekst  
#### Utfordring  
Klassifiser elementer i kategorier som gis ved inferenstidspunktet. I eksemplet nedenfor gir vi både kategoriene og teksten som skal klassifiseres i prompten(*playground_reference). 

Kundespørsmål: Hei, en av tastene på laptop-tastaturet mitt ble nylig ødelagt, og jeg trenger en erstatning:

Klassifisert kategori:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generer nye produktnavn
#### Utfordring
Lag produktnavn basert på eksempelord. Her inkluderer vi i prompten informasjon om produktet vi skal generere navn for. Vi gir også et lignende eksempel for å vise mønsteret vi ønsker å få. Vi har også satt temperaturverdien høy for å øke tilfeldigheten og gi mer innovative svar.

Produktbeskrivelse: En hjemmemilkshake-maskin
Seed-ord: rask, sunn, kompakt.
Produktnavn: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Produktbeskrivelse: Et par sko som kan passe enhver skostørrelse.
Seed-ord: tilpasningsdyktig, passform, omni-passform.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referanser  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Beste praksis for finjustering av GPT-modeller for å klassifisere tekst](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# For mer hjelp  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Bidragsytere
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
